<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/11_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11 · Datasets from real traffic

Part 1 judged agents by reading the output and deciding it looked about right. That does not
survive contact with a second person, a second week, or a model upgrade.

An **evaluation** replaces that judgement with something repeatable: a dataset of examples, a
target to run them through, and evaluators that score the results. This lesson builds the first
piece — and builds it from the deployed agent's own traffic rather than from imagination.

**New in this lesson:** `client.list_runs`, filters, `create_dataset`, `create_examples`, splits,
and a rule that grows a dataset for you

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-11-datasets"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. One agent, many testers

Everyone in this room is evaluating the **same deployed agent** from lesson 10. You do not need
your own deployment to evaluate one — you need its URL and a key, both of which you have.

Look up the deployment rather than pasting a URL: the record also tells you which tracing project
it writes to, and you will need that in a moment.

In [ ]:
#@title Connect to the shared deployment (run me) { display-mode: "form" }
# --- snippet:remote_agent v1 ---
import httpx
from langgraph.pregel.remote import RemoteGraph

HOST_API = "https://api.host.langchain.com"


def find_deployment(name: str = "support-agent") -> dict:
    """The shared deployment's record, looked up by name."""
    response = httpx.get(
        f"{HOST_API}/v2/deployments",
        params={"name_contains": name},
        headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]},
        timeout=30,
    )
    response.raise_for_status()
    for record in response.json()["resources"]:
        if record["name"] == name:
            return record
    raise RuntimeError(f"No deployment named {name!r} in this workspace.")


DEPLOYMENT = find_deployment()

# Every attendee's calls land in this one project. That is the point: shared traffic.
TRAFFIC_PROJECT_ID = DEPLOYMENT["tracer_session_id"]

# "support" is the graph key from langgraph.json in lesson 10.
support = RemoteGraph("support", url=DEPLOYMENT["url"], api_key=key)


def last_text(result: dict) -> str:
    """The final reply. A deployment returns JSON, so messages are dicts."""
    content = result["messages"][-1].get("content") or ""
    if isinstance(content, list):
        return " ".join(part.get("text", "") for part in content if isinstance(part, dict))
    return str(content)


def tool_names(result: dict) -> list[str]:
    """Every tool the run called, in order."""
    return [call["name"]
            for message in result["messages"]
            for call in (message.get("tool_calls") or [])]
# --- /snippet ---

print(f"{DEPLOYMENT['name']}: {DEPLOYMENT['status']}")

Two ids came back. `DEPLOYMENT["url"]` is where you send requests. `TRAFFIC_PROJECT_ID` is where
the deployment writes its traces — **its** project, not yours. That distinction matters: the agent
runs on the platform, so the platform does the tracing.

In [ ]:
from langsmith import Client

client = Client()

---

## 2. Make some traffic

Evaluating starts with a question you already have. Ask the agent a few things, including at least
one you suspect it gets wrong.

In [ ]:
questions = [
    "Ticket T-6: the laptop stand on order 1047 wobbles. What can we offer?",
    "Order 1042 arrived with a cracked leg. The customer wants a full refund.",
    "I ordered the wrong colour chair, order 1045. Can I swap it?",
]

for question in questions:
    result = support.invoke({"messages": [{"role": "user", "content": question}]})
    print("Q:", question)
    print("A:", last_text(result)[:220].replace("\n", " "), "\n")

Order 1047 was delivered 62 days ago, so the refund policy allows a **repair only**. Order 1042 is
damage on arrival, which is a full refund with no time limit. Order 1045 is a customer-error
exchange — 14 days, 10% restocking fee.

Read those three answers against those three rules. At least one is usually arguable. That
disagreement is the raw material for a dataset.

---

## 3. Find the runs

Your calls are now traces in the deployment's project, mixed in with everyone else's. Pull them
back out with `list_runs`.

In [ ]:
runs = list(client.list_runs(
    project_id=TRAFFIC_PROJECT_ID,
    is_root=True,          # the whole agent run, not the model calls inside it
    limit=10,
))

for run in runs:
    first = run.inputs.get("messages", [{}])[0]
    print(f"{str(run.start_time)[:19]}  {run.status:8}  {str(first.get('content'))[:70]}")

`is_root=True` is the important argument. Without it you get every span — each model call, each
tool call — and the agent run you actually want is buried among them.

Filtering is a query language rather than a set of keyword arguments, which is what lets you ask
the interesting questions:

```python
# runs that errored
filter='eq(status, "error")'

# slow runs
filter='gt(latency, 20)'

# runs a human scored badly
filter='and(eq(feedback_key, "correctness"), eq(feedback_score, 0))'

# runs that mention a specific order anywhere in the trace
tree_filter='search("1047")'
```

The last one is worth dwelling on. `filter` matches the root run; `tree_filter` matches **any span
in the trace**. "Find me the runs where the agent read the refund policy" is a `tree_filter`
question, and it is the kind you will actually ask when hunting a bug.

In [ ]:
# The runs worth testing are rarely the average ones. Start with the failures.
suspects = list(client.list_runs(
    project_id=TRAFFIC_PROJECT_ID,
    is_root=True,
    filter='or(eq(status, "error"), gt(latency, 30))',
    limit=5,
))

print(f"{len(suspects)} suspect runs")
for run in suspects:
    print(f"  {run.status:8} {run.latency}s  {run.error or ''}"[:110])

---

## 4. From run to example

An **example** is an input plus, optionally, a reference output. A **dataset** is a named,
versioned collection of them.

The inputs have to be shaped like something you can call the agent with — because that is exactly
what an experiment will do. For this agent that means `{"messages": [...]}`.

In [ ]:
import getpass

# Your own dataset. Everyone shares the agent; nobody shares your dataset.
me = getpass.getuser()
dataset = client.create_dataset(
    f"refund-decisions-{me}",
    description="Support questions where the refund policy gives a clear right answer.",
)
print(dataset.name, dataset.id)

In [ ]:
# Pulled from the traffic above, with the *correct* answer written by a human who read the policy.
examples = [
    {
        "inputs": {"messages": [{"role": "user", "content":
            "Ticket T-6: the laptop stand on order 1047 wobbles. What can we offer?"}]},
        "outputs": {"decision": "repair",
                    "because": "Faulty after 30 days (delivered 62 days ago) is repair only."},
    },
    {
        "inputs": {"messages": [{"role": "user", "content":
            "Order 1042 arrived with a cracked leg. The customer wants a full refund."}]},
        "outputs": {"decision": "refund",
                    "because": "Damaged on arrival: full refund or replacement, no time limit."},
    },
    {
        "inputs": {"messages": [{"role": "user", "content":
            "I ordered the wrong colour chair, order 1045. Can I swap it?"}]},
        "outputs": {"decision": "exchange",
                    "because": "Customer error: exchange within 14 days, 10% restocking fee."},
    },
    {
        "inputs": {"messages": [{"role": "user", "content":
            "Order 1046 was cancelled but I was charged anyway."}]},
        "outputs": {"decision": "escalate",
                    "because": "Cancelled but charged: refund within 5 days and escalate."},
    },
]

client.create_examples(dataset_id=dataset.id, examples=examples)
print(f"{len(examples)} examples")

### The reference output is a decision, not a transcript

Notice what the reference outputs are *not*: they are not the agent's answer, and they are not a
paragraph of ideal prose. They are the **thing you actually care about being right** — a decision
and the policy line behind it.

This is the single most consequential choice in an eval suite. Write references as full prose and
every evaluator you build is forced to compare wordings, which is expensive, noisy, and mostly
measures style. Write references as decisions and you can check them exactly.

If you find yourself unable to state the reference output crisply, that is usually a sign the
example is a bad test rather than a hard one.

---

## 5. Splits and versions

One dataset, several jobs: a handful of examples fast enough to run on every commit, and a longer
tail you run nightly. **Splits** carve out subsets without creating a second dataset.

In [ ]:
stored = list(client.list_examples(dataset_id=dataset.id))

client.update_examples(
    dataset_id=dataset.id,
    updates=[
        {"id": stored[0].id, "split": ["smoke"]},
        {"id": stored[1].id, "split": ["smoke"]},
        {"id": stored[2].id, "split": ["nightly"]},
        {"id": stored[3].id, "split": ["nightly"]},
    ],
)

smoke = list(client.list_examples(dataset_id=dataset.id, splits=["smoke"]))
print(f"smoke split: {len(smoke)} examples")

Datasets are also **versioned**. Every write creates a new version, and you can read the dataset as
it was at a point in time:

```python
client.list_examples(dataset_id=dataset.id, as_of="2026-08-21T12:00:00Z")
```

That is what makes a regression comparison honest. If a score drops between Monday and Friday, the
first question is always "did the agent get worse, or did the dataset change?" — and `as_of`
answers it.

---

## 6. Let LangSmith grow the dataset

Curating by hand does not scale past the first afternoon. A **rule** watches a tracing project and
adds matching runs to a dataset automatically.

Rules are the same machinery you will use for evaluators in the next two lessons, so it is worth
meeting them here where they only do one simple thing.

In [ ]:
rule = httpx.post(
    "https://api.smith.langchain.com/api/v1/runs/rules",
    headers={"x-api-key": key},
    json={
        "display_name": f"collect-errors-{me}",
        "session_id": TRAFFIC_PROJECT_ID,       # watch the deployment's traffic
        "filter": 'eq(status, "error")',        # only the failures
        "sampling_rate": 1.0,                   # all of them
        "add_to_dataset_id": str(dataset.id),   # land them here
        "is_enabled": True,
    },
    timeout=30,
).json()

print("rule:", rule["display_name"], rule["id"])

From now on, every errored run in that project becomes an example in your dataset, with no one
remembering to do it. Swap the filter and the same mechanism gives you a different corpus:

| `filter` | The dataset you end up with |
|---|---|
| `eq(status, "error")` | crashes — always worth keeping |
| `eq(feedback_score, 0)` on a thumbs-down key | what users complained about |
| `gt(latency, 30)` | the slow tail |
| `search("refund")` in `tree_filter` | one feature's traffic |

**Sampling is not decoration.** At 1.0 on a busy project you will collect thousands of
near-identical examples, which costs money to evaluate and teaches you nothing new. Production
rules usually sit somewhere between 0.01 and 0.1 for volume traffic, and at 1.0 only for rare
events like errors.

You can delete the rule when you are done with it:

```python
httpx.delete(f"https://api.smith.langchain.com/api/v1/runs/rules/{rule['id']}",
             headers={"x-api-key": key})
```

---

## 📌 Key takeaways

- An evaluation is three parts: a **dataset**, a **target** to run it through, and **evaluators** to score the results.
- The best examples come from real traffic, not imagination — the agent has already found the cases you would not have thought of.
- A deployed agent traces to **its own** project, so that is where you go looking for its runs.
- `is_root=True` gets you agent runs instead of every span inside them.
- `filter` matches the root run; `tree_filter` matches any span in the trace.
- Example inputs must be shaped like a real call, because an experiment will call the target with them verbatim.
- Write reference outputs as **decisions**, not prose — otherwise every evaluator ends up grading style.
- Splits let one dataset serve both a fast smoke suite and a slow nightly one.
- Datasets are versioned, so `as_of` answers "did the agent regress, or did the dataset change?"
- A rule with `add_to_dataset_id` grows a dataset from live traffic; the `filter` decides what kind of dataset you get.

---

## ➡️ Next

**[12 · Evaluators that live in LangSmith](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/12_evaluators.ipynb)**

You have examples and a target. The missing piece is the grader — and rather than writing it in
this notebook, you will create it **inside LangSmith**, where it outlives the kernel and can be
pointed at anything.